# Run MoneyPrinterTurbo on Google Colab

This notebook installs [MoneyPrinterTurbo](https://github.com/harry0703/MoneyPrinterTurbo) in an isolated Python 3.11 environment and launches its WebUI. Colab runtimes are temporary, so generated files are removed when the runtime is reset.

**What you get**

- The official MoneyPrinterTurbo checkout under `/content/MoneyPrinterTurbo`
- Dependencies installed with `uv` into a project virtualenv (Colab's global packages are left alone)
- A public HTTPS URL to the Streamlit WebUI via [ngrok](https://ngrok.com/)

**Before you start**

1. Runtime → Change runtime type → keep the default CPU runtime (a GPU is optional).
2. Create a free [ngrok](https://dashboard.ngrok.com/signup) account and copy your [authtoken](https://dashboard.ngrok.com/get-started/your-authtoken).
3. After the WebUI opens, configure an LLM provider and a footage source (Pexels, Pixabay, or Coverr) under **Settings**. Edge TTS works without an API key.

Run the cells in order. Each install and launch cell is safe to rerun.


## 1. Install MoneyPrinterTurbo

The setup is safe to run again: it clones the repository on the first run and updates it on later runs. Dependencies are installed in the project's virtual environment to avoid conflicts with Colab's preinstalled packages.


In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

REPO_DIR = Path("/content/MoneyPrinterTurbo")
REPO_URL = "https://github.com/harry0703/MoneyPrinterTurbo.git"

# Colab images already include ffmpeg. ImageMagick is only needed for a few
# MoviePy text paths; install it when missing so subtitle rendering stays reliable.
missing_apt_packages = []
if shutil.which("ffmpeg") is None:
    missing_apt_packages.append("ffmpeg")
if shutil.which("convert") is None and shutil.which("magick") is None:
    missing_apt_packages.append("imagemagick")
if missing_apt_packages:
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(
        ["apt-get", "install", "-y", "-qq", *missing_apt_packages], check=True
    )

# Update an existing checkout so rerunning this cell does not fail during clone.
if (REPO_DIR / ".git").is_dir():
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True
    )
elif REPO_DIR.exists():
    raise RuntimeError(f"{REPO_DIR} exists but is not a Git repository")
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True
    )

os.chdir(REPO_DIR)
subprocess.run(["python", "-m", "pip", "install", "-q", "uv", "pyngrok"], check=True)
subprocess.run(["uv", "python", "install", "3.11"], check=True)
# Use the lockfile in an isolated environment without changing Colab's preinstalled packages.
subprocess.run(["uv", "sync", "--frozen", "--python", "3.11"], check=True)
print(f"MoneyPrinterTurbo is ready in {REPO_DIR}")


## 2. Configure the Colab tunnel

MoneyPrinterTurbo itself does not require ngrok for normal local use. This notebook uses ngrok only because the Streamlit server runs inside a remote Colab runtime.

Create an account and copy your authentication token from the [ngrok dashboard](https://dashboard.ngrok.com/get-started/your-authtoken). The next cell reads it without displaying or storing it in the notebook.


In [ ]:
from getpass import getpass

from pyngrok import ngrok

# Close tunnels from earlier runs before configuring a new one.
ngrok.kill()

ngrok_token = getpass("Enter your ngrok authentication token: ").strip()
if not ngrok_token:
    raise ValueError("An ngrok authentication token is required")
ngrok.set_auth_token(ngrok_token)
del ngrok_token
print("ngrok authentication configured")


## 3. Launch the WebUI

The cell waits for Streamlit to become healthy before creating the public URL. If startup fails, it includes the recent server log in the error. After opening the URL, use **Settings** to configure the required model and media API keys.


In [ ]:
import time
from urllib.error import URLError
from urllib.request import urlopen

PORT = 8501
LOG_PATH = Path("/content/moneyprinterturbo-webui.log")

# Make this cell safe to rerun by closing the previous process, log handle, and tunnel.
previous_streamlit_proc = globals().get("streamlit_proc")
if previous_streamlit_proc is not None and previous_streamlit_proc.poll() is None:
    previous_streamlit_proc.terminate()
    try:
        previous_streamlit_proc.wait(timeout=10)
    except subprocess.TimeoutExpired:
        previous_streamlit_proc.kill()
        previous_streamlit_proc.wait(timeout=5)
previous_streamlit_log = globals().get("streamlit_log")
if previous_streamlit_log is not None and not previous_streamlit_log.closed:
    previous_streamlit_log.close()
ngrok.kill()

streamlit_log = LOG_PATH.open("w", encoding="utf-8")
streamlit_proc = subprocess.Popen(
    [
        "uv",
        "run",
        "streamlit",
        "run",
        "webui/Main.py",
        f"--server.port={PORT}",
        "--server.address=0.0.0.0",
        "--browser.gatherUsageStats=False",
        "--client.toolbarMode=minimal",
        "--server.showEmailPrompt=False",
        "--server.enableCORS=True",
        "--server.headless=true",
    ],
    cwd=REPO_DIR,
    stdout=streamlit_log,
    stderr=subprocess.STDOUT,
    text=True,
    env={
        **os.environ,
        "PYTHONPATH": str(REPO_DIR),
        "BROWSER": "none",
    },
)

# Poll the health endpoint because startup time varies across Colab runtimes.
deadline = time.time() + 90
server_ready = False
while time.time() < deadline:
    if streamlit_proc.poll() is not None:
        break
    try:
        with urlopen(f"http://127.0.0.1:{PORT}/_stcore/health", timeout=2) as response:
            server_ready = response.status == 200
    except (URLError, TimeoutError):
        pass
    if server_ready:
        break
    time.sleep(2)

if not server_ready:
    streamlit_log.flush()
    recent_log = LOG_PATH.read_text(encoding="utf-8", errors="replace")[-4000:]
    raise RuntimeError(f"Streamlit failed to start. Recent log:\n{recent_log}")

# Use an explicit IPv4 URL because pyngrok may otherwise route localhost through ::1.
public_tunnel = ngrok.connect(addr=f"http://127.0.0.1:{PORT}", proto="http", bind_tls=True)
print("MoneyPrinterTurbo is ready:")
print(public_tunnel.public_url)
print(f"Server log: {LOG_PATH}")


## After the WebUI opens

1. Open the printed `https://….ngrok-free.app` URL.
2. If ngrok shows an interstitial page, click **Visit Site**.
3. In the WebUI **Settings** panel, set:
   - an **LLM provider** and API key (OpenAI, Gemini, DeepSeek, Moonshot, Groq, and others are supported)
   - a **video source** API key from [Pexels](https://www.pexels.com/api/), [Pixabay](https://pixabay.com/api/docs/), or [Coverr](https://coverr.co/developers), unless you upload local footage
4. Enter a topic, pick voice / subtitle / aspect-ratio options, and generate.

Generated videos land in `/content/MoneyPrinterTurbo/storage/`. Download them from the WebUI or copy them out before the Colab runtime is recycled.

Keep this notebook session running while you use the WebUI. Stopping the runtime, resetting it, or letting it idle-timeout also stops Streamlit and the ngrok tunnel.


## Optional: keep videos on Google Drive

Colab disks are wiped on reset. Run this cell after a successful generation to copy `storage/` onto your Drive.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

source_dir = REPO_DIR / "storage"
target_dir = Path("/content/drive/MyDrive/MoneyPrinterTurbo/storage")
if not source_dir.exists():
    raise FileNotFoundError(f"Nothing to copy: {source_dir} does not exist yet")

target_dir.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(source_dir, target_dir, dirs_exist_ok=True)
print(f"Copied {source_dir} -> {target_dir}")


## Optional: stop the WebUI

Run this when you are finished, or before relaunching with different options. The launch cell already stops a previous server, so this is only needed for a clean shutdown.


In [ ]:
previous_streamlit_proc = globals().get("streamlit_proc")
if previous_streamlit_proc is not None and previous_streamlit_proc.poll() is None:
    previous_streamlit_proc.terminate()
    try:
        previous_streamlit_proc.wait(timeout=10)
    except subprocess.TimeoutExpired:
        previous_streamlit_proc.kill()
        previous_streamlit_proc.wait(timeout=5)
    print("Stopped the Streamlit process")
else:
    print("No running Streamlit process")

previous_streamlit_log = globals().get("streamlit_log")
if previous_streamlit_log is not None and not previous_streamlit_log.closed:
    previous_streamlit_log.close()

ngrok.kill()
print("Closed ngrok tunnels")
